## Modélisation

In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
PROJECT_DIR = Path(r"C:\Users\imane\Desktop\Fil-rouge")
GOLD_DIR = PROJECT_DIR / "data" / "gold"
STAR_DIR = GOLD_DIR / "star_schema"
STAR_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = GOLD_DIR / "f1_final_enriched.csv"

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(INPUT_PATH)

print("Shape input:", df.shape)
print("Columns:", df.columns.tolist())

# ============================================================
# CLEAN MINIMAL
# ============================================================
df = df.copy()

# harmonisation texte utile
for col in ["driverId", "driverName", "driverCode", "constructorId", "constructorName", "circuit", "gp", "session"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# ============================================================
# 1) DIM_DRIVER
# ============================================================
driver_cols = [c for c in ["driverId", "driverName", "driverCode"] if c in df.columns]

dim_driver = (
    df[driver_cols]
    .drop_duplicates()
    .sort_values("driverId")
    .reset_index(drop=True)
)

# ============================================================
# 2) DIM_CONSTRUCTOR
# ============================================================
constructor_cols = [c for c in ["constructorId", "constructorName"] if c in df.columns]

dim_constructor = (
    df[constructor_cols]
    .drop_duplicates()
    .sort_values("constructorId")
    .reset_index(drop=True)
)

# ============================================================
# 3) DIM_CIRCUIT
# ============================================================

df["circuit"] = df["circuit"].astype(str).str.strip()
df["gp"] = df["gp"].astype(str).str.strip()
df["circuitKey"] = df["gp"] + "_" + df["circuit"]

dim_circuit = (
    df[["circuitKey", "circuit", "gp"]]
    .drop_duplicates()
    .sort_values(["gp", "circuit"])
    .reset_index(drop=True)
)

# ============================================================
# 4) DIM_TIME
# ============================================================
time_cols = [c for c in ["season", "round"] if c in df.columns]

dim_time = (
    df[time_cols]
    .drop_duplicates()
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

# clé temps simple pour Power BI
dim_time["timeKey"] = (
    dim_time["season"].astype(str) + "_" + dim_time["round"].astype(str)
)

# réordonner
dim_time = dim_time[["timeKey", "season", "round"]]

# ============================================================
# 5) DIM_SESSION
# ============================================================
session_cols = [c for c in ["session"] if c in df.columns]

dim_session = (
    df[session_cols]
    .drop_duplicates()
    .sort_values("session")
    .reset_index(drop=True)
)

# ============================================================
# 6) FACT_PERFORMANCE
# ============================================================
# créer clé circuit unique
df["circuit"] = df["circuit"].astype(str).str.strip()
df["gp"] = df["gp"].astype(str).str.strip()
df["circuitKey"] = df["gp"] + "_" + df["circuit"]

fact_cols = [
    # clés
    "season", "round", "driverId", "constructorId", "circuitKey", "session",
   # attributs descriptifs utiles
    "circuit",

    # mesures brutes
    "bestLapTime_sec", "avgLapTime_sec", "stdLapTime_sec", "maxSpeed_kmh",
    "gridPosition", "finishPosition", "qualyPosition", "startPosition",
    "driverPoints", "constructorPoints",
    "estimatedSalaryUSD", "estimatedSalaryUSD_imputed", "costCapUSD",

    # KPI / features
    "raceGain",
    "consistencyIndex",
    "qualifyingPerformanceIndex",
    "topSpeedIndex",
    "driverROI",
    "driverEfficiency",
    "budgetEfficiency",
    "pointsPerMillionUSD",
    "speedEfficiency",
    "isPodium",
    "teamPodiums",
    "costPerPodium",
    "teammateDelta",
    "avgTeammateDelta_season",
    "paceAdvantage",
    "speed_zscore",
    "pointsRankInTeam",
    "qualiToRaceDelta",
    "avgFinishPosition_season",
    "avgQualyPosition_season",
    "avgRaceGain_season",
    "avgConsistency_season",
    "avgTopSpeed_season",
    "driverPerformanceScore",
    "driverPerformanceScore_scaled"
]

fact_cols = [c for c in fact_cols if c in df.columns]

fact_performance = df[fact_cols].copy()

# timeKey pour relation facile avec dim_time
fact_performance["timeKey"] = (
    fact_performance["season"].astype(str) + "_" + fact_performance["round"].astype(str)
)

# ordre colonnes recommandé
front_cols = ["timeKey", "season", "round", "driverId", "constructorId", "circuit", "session"]
other_cols = [c for c in fact_performance.columns if c not in front_cols]
fact_performance = fact_performance[front_cols + other_cols]

# enlever éventuels doublons de la clé métier
fact_performance = (
    fact_performance
    .drop_duplicates(subset=["season", "round", "driverId", "session"])
    .reset_index(drop=True)
)

# ============================================================
# 7) EXPORT
# ============================================================
dim_driver.to_csv(STAR_DIR / "dim_driver.csv", index=False, encoding="utf-8")
dim_constructor.to_csv(STAR_DIR / "dim_constructor.csv", index=False, encoding="utf-8")
dim_circuit.to_csv(STAR_DIR / "dim_circuit.csv", index=False, encoding="utf-8")
dim_time.to_csv(STAR_DIR / "dim_time.csv", index=False, encoding="utf-8")
dim_session.to_csv(STAR_DIR / "dim_session.csv", index=False, encoding="utf-8")
fact_performance.to_csv(STAR_DIR / "fact_performance.csv", index=False, encoding="utf-8")

# ============================================================
# 8) CHECK FINAL
# ============================================================
print("\n✅ STAR SCHEMA CREATED")
print("Export folder:", STAR_DIR)

print("\nDIM_DRIVER:", dim_driver.shape)
print("DIM_CONSTRUCTOR:", dim_constructor.shape)
print("DIM_CIRCUIT:", dim_circuit.shape)
print("DIM_TIME:", dim_time.shape)
print("DIM_SESSION:", dim_session.shape)
print("FACT_PERFORMANCE:", fact_performance.shape)

print("\nPreview FACT:")
print(fact_performance.head())

print("\nMissing values in fact keys:")
print(
    fact_performance[["timeKey", "season", "round", "driverId", "constructorId", "circuit", "session"]]
    .isna()
    .sum()
)

Shape input: (4129, 54)
Columns: ['season', 'round', 'gp', 'session', 'circuit', 'driverCode', 'bestLapTime_sec', 'avgLapTime_sec', 'stdLapTime_sec', 'maxSpeed_kmh', 'gridPosition', 'finishPosition', 'qualyPosition', 'startPosition', 'driverId', 'driverName', 'constructorId', 'driverRank', 'driverPoints', 'salary_raw', 'estimatedSalaryUSD', 'salarySourceURL', 'salaryMissingFlag', 'salaryCoverageStatus', 'estimatedSalaryUSD_imputed', 'constructorName', 'constructorRank', 'constructorPoints', 'costCapUSD', 'raceGain', 'consistencyIndex', 'qualifyingPerformanceIndex', 'topSpeedIndex', 'driverROI', 'driverEfficiency', 'budgetEfficiency', 'pointsPerMillionUSD', 'speedEfficiency', 'isPodium', 'teamPodiums', 'costPerPodium', 'teammateDelta', 'avgTeammateDelta_season', 'paceAdvantage', 'speed_zscore', 'pointsRankInTeam', 'qualiToRaceDelta', 'avgFinishPosition_season', 'avgQualyPosition_season', 'avgRaceGain_season', 'avgConsistency_season', 'avgTopSpeed_season', 'driverPerformanceScore', 'driv